In [83]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import re
import numpy as np
import pandas as pd

from scipy.stats import chi2_contingency, pointbiserialr

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.dummy import DummyClassifier

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

In [84]:
DATA_PATH = Path("../data/processed/nhanes_merged_adults_final.csv")
OUTPUT_DIR = Path("../data/processed/model_outputs_prediabetes")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "prediabetes"
RANDOM_STATE = 42
TEST_SIZE = 0.20

print("DATA_PATH:", DATA_PATH.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

DATA_PATH: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\nhanes_merged_adults_final.csv
OUTPUT_DIR: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_prediabetes


In [85]:
[c for c in df.columns if "age" in c.lower() or "gender" in c.lower() or "sex" in c.lower()]

['age_years',
 'gender',
 'bpd035___age_told_had_hypertension',
 'did040___age_when_first_told_you_had_diabetes',
 'kiq005___how_often_have_urinary_leakage?',
 'kiq050___how_much_did_urine_leakage_bother_you?',
 'mcq025___age_when_first_had_asthma',
 'mcq151___age_in_years_at_first_menstrual_period',
 'mcd180b___age_when_told_you_had_chf',
 'mcd180c___age_when_told_you_had_chd',
 'mcd180d___age_when_told_you_had_angina',
 'mcd180e___age_when_told_you_had_heart_attack',
 'mcd180f___age_when_told_you_had_a_stroke',
 'mcd180m___age_when_told_you_had_thyroid_problem',
 'mcd180l___age_when_told_you_had_liver_condition',
 'mcq570___age_when_1st_had_gallbladder_surgery?',
 'rhq010___age_when_first_menstrual_period_occurred',
 'rhq020___age_range_at_first_menstrual_period',
 'rhq060___age_at_last_menstrual_period',
 'rhq070___age_range_at_last_menstrual_period',
 'rhd180___age_at_first_live_birth',
 'rhd190___age_at_last_live_birth',
 'rhq332___age_when_both_ovaries_removed',
 'smd030___age_st

In [86]:
df = pd.read_csv(DATA_PATH, low_memory=False)

print("Shape:", df.shape)
print("First 10 columns:")
print(df.columns[:10].tolist())

assert TARGET in df.columns, f"{TARGET} not found in dataframe."

print(df[TARGET].value_counts(dropna=False).sort_index())
print()
print("Target prevalence:")
print(df[TARGET].value_counts(normalize=True, dropna=False).sort_index().round(4))

Shape: (7437, 878)
First 10 columns:
['SEQN', 'age_years', 'income_poverty_ratio', 'mec_exam_weight', 'interview_weight', 'survey_psu', 'survey_stratum', 'gender', 'ethnicity', 'education']
prediabetes
0    6755
1     682
Name: count, dtype: int64

Target prevalence:
prediabetes
0    0.9083
1    0.0917
Name: proportion, dtype: float64


# Sleeping Features

In [87]:
SLEEP_COLS = [
    "sld012___sleep_hours___weekdays_or_workdays",
    "sld013___sleep_hours___weekends",
    "slq300___usual_sleep_time_on_weekdays_or_workdays",
    "slq310___usual_wake_time_on_weekdays_or_workdays",
    "slq320___usual_sleep_time_on_weekends",
    "slq330___usual_wake_time_on_weekends",
    "slq030___how_often_do_you_snore?",
]

sleep_df = df[SLEEP_COLS].copy()
sleep_df.head()

,sld012___sleep_hours___weekdays_or_workdays,sld013___sleep_hours___weekends,slq300___usual_sleep_time_on_weekdays_or_workdays,slq310___usual_wake_time_on_weekdays_or_workdays,slq320___usual_sleep_time_on_weekends,slq330___usual_wake_time_on_weekends,slq030___how_often_do_you_snore?
0,7.5,8.0,22:00,05:30,23:00,07:00,1.0
1,8.0,8.0,00:00,08:00,03:00,11:00,0.0
2,8.5,8.0,22:00,06:30,23:00,07:00,0.0
3,10.0,13.0,23:00,09:00,23:00,12:00,0.0
4,6.5,8.0,08:00,14:35,21:00,05:00,0.0


In [88]:
sleep_overview = pd.DataFrame({
    "feature": sleep_df.columns,
    "dtype": sleep_df.dtypes.astype(str).values,
    "missing_pct": (sleep_df.isna().mean() * 100).round(2).values,
    "n_unique": [sleep_df[c].nunique(dropna=True) for c in sleep_df.columns]
}).sort_values("missing_pct", ascending=False)

sleep_overview

,feature,dtype,missing_pct,n_unique
1,sld013___sleep_hours___weekends,float64,0.86,24
0,sld012___sleep_hours___weekdays_or_workdays,float64,0.71,24
2,slq300___usual_sleep_time_on_weekdays_or_workdays,str,0.65,65
3,slq310___usual_wake_time_on_weekdays_or_workdays,str,0.62,113
4,slq320___usual_sleep_time_on_weekends,str,0.59,59
5,slq330___usual_wake_time_on_weekends,str,0.58,70
6,slq030___how_often_do_you_snore?,float64,0.00,6


In [89]:
def get_example_values(series, n=10):
    vals = series.dropna().unique()[:n]
    return list(vals)

sleep_examples = pd.DataFrame({
    "feature": sleep_df.columns,
    "example_values": [get_example_values(sleep_df[c], n=10) for c in sleep_df.columns]
})

sleep_examples

,feature,example_values
0,sld012___sleep_hours___weekdays_or_workdays,"[7.5, 8.0, 8.5, 10.0, 6.5, 9.5, 11.0, 4.5, 7.0..."
1,sld013___sleep_hours___weekends,"[8.0, 13.0, 10.0, 9.0, 4.5, 6.5, 7.0, 14.0, 11..."
2,slq300___usual_sleep_time_on_weekdays_or_workdays,"[22:00, 00:00, 23:00, 08:00, 01:00, 21:00, 18:..."
3,slq310___usual_wake_time_on_weekdays_or_workdays,"[05:30, 08:00, 06:30, 09:00, 14:35, 10:30, 06:..."
4,slq320___usual_sleep_time_on_weekends,"[23:00, 03:00, 21:00, 22:00, 00:00, 02:00, 01:..."
5,slq330___usual_wake_time_on_weekends,"[07:00, 11:00, 12:00, 05:00, 08:00, 09:00, 01:..."
6,slq030___how_often_do_you_snore?,"[1.0, 0.0, 2.0, 3.0, 9.0, 7.0]"


In [90]:
for col in [
    "sld012___sleep_hours___weekdays_or_workdays",
    "sld013___sleep_hours___weekends",
]:
    print("\n" + "=" * 80)
    print(col)
    print(sleep_df[col].value_counts(dropna=False).sort_index().head(50))


sld012___sleep_hours___weekdays_or_workdays
sld012___sleep_hours___weekdays_or_workdays
2.0       38
3.0       51
3.5       32
4.0      148
4.5       67
5.0      294
5.5      182
6.0      647
6.5      508
7.0     1283
7.5      723
8.0     1328
8.5      476
9.0      790
9.5      180
10.0     314
10.5      58
11.0     151
11.5      17
12.0      55
12.5       3
13.0      19
13.5       3
14.0      17
NaN       53
Name: count, dtype: int64

sld013___sleep_hours___weekends
sld013___sleep_hours___weekends
2.0       19
3.0       29
3.5       19
4.0       85
4.5       39
5.0      215
5.5       70
6.0      436
6.5      190
7.0      867
7.5      349
8.0     1391
8.5      433
9.0     1359
9.5      259
10.0     800
10.5     100
11.0     391
11.5      32
12.0     158
12.5      18
13.0      66
13.5       4
14.0      44
NaN       64
Name: count, dtype: int64


In [91]:
sleep_df[[
    "sld012___sleep_hours___weekdays_or_workdays",
    "sld013___sleep_hours___weekends",
]].describe()

,sld012___sleep_hours___weekdays_or_workdays,sld013___sleep_hours___weekends
count,7384.000000,7373.000000
mean,7.531352,8.351078
std,1.664477,1.813858
min,2.000000,2.000000
25%,6.500000,7.000000
50%,7.500000,8.000000
75%,8.500000,9.500000
max,14.000000,14.000000


In [92]:
sleep_duration_compare = pd.DataFrame({
    "weekday_sleep_hours": sleep_df["sld012___sleep_hours___weekdays_or_workdays"],
    "weekend_sleep_hours": sleep_df["sld013___sleep_hours___weekends"],
})

sleep_duration_compare["weekend_minus_weekday"] = (
    sleep_duration_compare["weekend_sleep_hours"] - sleep_duration_compare["weekday_sleep_hours"]
)

sleep_duration_compare.describe()

,weekday_sleep_hours,weekend_sleep_hours,weekend_minus_weekday
count,7384.000000,7373.000000,7354.000000
mean,7.531352,8.351078,0.820914
std,1.664477,1.813858,1.736213
min,2.000000,2.000000,-9.000000
25%,6.500000,7.000000,0.000000
50%,7.500000,8.000000,0.500000
75%,8.500000,9.500000,2.000000
max,14.000000,14.000000,10.000000


In [93]:
time_cols = [
    "slq300___usual_sleep_time_on_weekdays_or_workdays",
    "slq310___usual_wake_time_on_weekdays_or_workdays",
    "slq320___usual_sleep_time_on_weekends",
    "slq330___usual_wake_time_on_weekends",
]

for col in time_cols:
    print("\n" + "=" * 80)
    print(col)
    print(sleep_df[col].value_counts(dropna=False).head(30))


slq300___usual_sleep_time_on_weekdays_or_workdays
slq300___usual_sleep_time_on_weekdays_or_workdays
23:00    1507
22:00    1446
00:00     959
21:00     669
22:30     473
01:00     396
23:30     308
02:00     281
21:30     275
20:00     208
03:00     129
20:30      80
00:30      74
04:00      61
NaN        48
99999      46
01:30      43
19:00      38
08:00      37
09:00      35
05:00      32
10:00      29
02:30      26
06:00      25
18:00      21
19:30      20
07:00      19
08:30      12
11:00      12
12:00      11
Name: count, dtype: int64

slq310___usual_wake_time_on_weekdays_or_workdays
slq310___usual_wake_time_on_weekdays_or_workdays
06:00    1277
07:00     957
05:00     842
06:30     576
08:00     537
05:30     483
09:00     323
04:00     297
07:30     266
10:00     237
04:30     226
08:30     110
03:00     106
11:00     105
12:00      91
05:45      74
06:15      61
06:45      58
03:30      57
09:30      49
NaN        46
05:15      38
15:00      37
02:00      35
13:00      34
14:0

In [94]:
import re

def is_hhmm(x):
    if pd.isna(x):
        return True
    x = str(x).strip()
    return bool(re.match(r"^\d{1,2}:\d{2}$", x))

time_format_check = pd.DataFrame({
    "feature": time_cols,
    "invalid_format_n": [sleep_df[~sleep_df[c].apply(is_hhmm)][c].notna().sum() for c in time_cols],
    "invalid_examples": [
        sleep_df.loc[~sleep_df[c].apply(is_hhmm), c].dropna().astype(str).unique()[:10].tolist()
        for c in time_cols
    ]
})

time_format_check

,feature,invalid_format_n,invalid_examples
0,slq300___usual_sleep_time_on_weekdays_or_workdays,48,"[99999, 77777]"
1,slq310___usual_wake_time_on_weekdays_or_workdays,26,[99999]
2,slq320___usual_sleep_time_on_weekends,49,"[99999, 77777]"
3,slq330___usual_wake_time_on_weekends,36,"[99999, 77777]"


In [95]:
import numpy as np

# -------------------------
# Clean invalid NHANES codes
# -------------------------

TIME_COLS = [
    "slq300___usual_sleep_time_on_weekdays_or_workdays",
    "slq310___usual_wake_time_on_weekdays_or_workdays",
    "slq320___usual_sleep_time_on_weekends",
    "slq330___usual_wake_time_on_weekends",
]

for col in TIME_COLS:
    df[col] = df[col].replace(["77777", "99999", 77777, 99999], np.nan)

# -------------------------
# Convert HH:MM → minutes
# -------------------------

def hhmm_to_minutes(x):

    if pd.isna(x):
        return np.nan

    try:
        h, m = str(x).split(":")
        return int(h)*60 + int(m)
    except:
        return np.nan


df["weekday_bedtime_min"] = df[
    "slq300___usual_sleep_time_on_weekdays_or_workdays"
].apply(hhmm_to_minutes)

df["weekday_waketime_min"] = df[
    "slq310___usual_wake_time_on_weekdays_or_workdays"
].apply(hhmm_to_minutes)


# -------------------------
# Circular encoding
# -------------------------

def circular_encode(series, period=1440):

    radians = 2 * np.pi * series / period

    return np.sin(radians), np.cos(radians)


df["sin_weekday_bedtime"], df["cos_weekday_bedtime"] = circular_encode(
    df["weekday_bedtime_min"]
)

df["sin_weekday_wake"], df["cos_weekday_wake"] = circular_encode(
    df["weekday_waketime_min"]
)


# -------------------------
# Sleep duration
# -------------------------

df["sleep_hours_weekday"] = df[
    "sld012___sleep_hours___weekdays_or_workdays"
]

df["sleep_hours_weekend"] = df[
    "sld013___sleep_hours___weekends"
]


# -------------------------
# Social jetlag
# -------------------------

df["social_jetlag"] = (
    df["sleep_hours_weekend"] -
    df["sleep_hours_weekday"]
)

# Broad Model

In [96]:
overview = pd.DataFrame({
    "feature": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_pct": df.isna().mean().values * 100,
    "n_unique": [df[c].nunique(dropna=True) for c in df.columns]
}).sort_values(["missing_pct", "n_unique"], ascending=[False, True])

overview.head(30)

,feature,dtype,missing_pct,n_unique
166,mcq149___menstrual_periods_started_yet?,float64,100.000000,0
167,mcq151___age_in_years_at_first_menstrual_period,float64,100.000000,0
294,smq621___cigarettes_smoked_in_entire_life,float64,100.000000,0
295,smd630___age_first_smoked_whole_cigarette,float64,100.000000,0
434,LBXIGG_cytomegalovirus_cmv_igg,float64,100.000000,0
435,LBXIGM_cytomegalovirus_cmv_igm,float64,100.000000,0
436,LBXIGGA_cytomegalovirus_cmv_igg_avidity,float64,100.000000,0
357,medication_21,str,99.973107,2
358,medication_22,str,99.973107,2
356,medication_20,str,99.959661,3


In [97]:
LEAKAGE_COLS = [
    TARGET,
    "diabetes",
    "diq010___doctor_told_you_have_diabetes",
    "diq160___ever_told_you_have_prediabetes",
    "rxd_disease_list",
]

QUESTIONNAIRE_PREFIXES = (
    "slq", "sld", "dpq", "paq", "huq", "mcq", "bpq", "kiq", "cdq", "smq", "alq",
    "whq", "rhq"
)

ALWAYS_INCLUDE_COLS = [
    "age_years",
    "gender",
    "med_count"
    "fasting_glucose_mg_dl",
    "triglycerides_mg_dl",
    "hdl_cholesterol_mg_dl",
    "total_cholesterol_mg_dl",
    "uacr_mg_g",

]

EXCLUDE_PATTERNS = [
    r"^seqn$",
    r"cluster",
    r"stratum",
    r"weight",
    r"sample_weight",
    r"survey_",
    r"^rxd",
    r"medication",
    r"drug",
    r"icd",
    r"diagnosis",
    r"lab",
    r"exam",
    r"fasting",
    r"liver_",
    r"bmi$",
    r"waist",
    r"hip",
    r"height",
    r"weight_kg",
    r"cholesterol",
    r"creatin",
    r"ferritin",
    r"vitamin",
    r"iron",
    r"protein",
    r"carbs",
    r"fat$",
    r"calories",
    r"pulse_",
    r"fatigue_binary",
    r"fatigue_score",
    r"glucose",
    r"insulin",
    r"hba1c",
    r"a1c",
    r"glyco",
]

MANUAL_DROP_COLS = [
    # bei Bedarf später ergänzen
]

In [98]:
def has_allowed_prefix(col: str, prefixes=QUESTIONNAIRE_PREFIXES) -> bool:
    col_low = col.lower()
    return col_low.startswith(prefixes)

def matches_any_pattern(col: str, patterns) -> bool:
    col_low = col.lower()
    return any(re.search(p, col_low) for p in patterns)

def is_binary_series(s: pd.Series) -> bool:
    vals = set(pd.Series(s.dropna()).unique())
    return vals.issubset({0, 1}) and len(vals) <= 2

def get_example_values(s: pd.Series, n=5):
    vals = s.dropna().unique()[:n]
    return list(vals)

def cramers_v(x: pd.Series, y: pd.Series) -> float:
    table = pd.crosstab(x, y)
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan
    chi2 = chi2_contingency(table)[0]
    n = table.values.sum()
    r, k = table.shape
    denom = min(k - 1, r - 1)
    if denom == 0 or n == 0:
        return np.nan
    return np.sqrt((chi2 / n) / denom)

def safe_pointbiserial(x: pd.Series, y: pd.Series):
    valid = x.notna() & y.notna()
    if valid.sum() < 30:
        return np.nan, np.nan
    try:
        r, p = pointbiserialr(x[valid], y[valid])
        return r, p
    except Exception:
        return np.nan, np.nan

def infer_feature_type(series: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(series):
        nunique = series.nunique(dropna=True)
        if nunique <= 10:
            return "categorical"
        return "numeric"
    return "categorical"

In [99]:
candidate_cols = []

for col in df.columns:
    if col == TARGET:
        continue
    if col in LEAKAGE_COLS:
        continue
    if col in MANUAL_DROP_COLS:
        continue
    if matches_any_pattern(col, EXCLUDE_PATTERNS):
        continue

    if col in ALWAYS_INCLUDE_COLS:
        candidate_cols.append(col)
        continue

    if not has_allowed_prefix(col):
        continue

    candidate_cols.append(col)

print("Number of initial candidate questionnaire features:", len(candidate_cols))
print(candidate_cols[:80])

Number of initial candidate questionnaire features: 142
['age_years', 'gender', 'triglycerides_mg_dl', 'uacr_mg_g', 'alq111___ever_had_a_drink_of_any_kind_of_alcohol', 'alq121___past_12_mo_how_often_drink_alcoholic_bev', 'alq130___avg_#_alcoholic_drinks/day___past_12_mos', 'alq142___#_days_have_4_or_5_drinks/past_12_mos', 'alq270___#_times_4_5_drinks_in_2hrs/past_12_mos', 'alq280___#_times_8+_drinks_in_1_day/past_12_mos', 'alq290___#_times_12+_drinks_in_1_day/past_12_mos', 'alq151___ever_have_4/5_or_more_drinks_every_day?', 'alq170___past_30_days_#_times_4_5_drinks_on_an_oc', 'bpq020___ever_told_you_had_high_blood_pressure', 'bpq030___told_had_high_blood_pressure___2+_times', 'bpq040a___taking_prescription_for_hypertension', 'bpq050a___now_taking_prescribed_medicine_for_hbp', 'bpq100d___now_taking_prescribed_medicine', 'cdq001___sp_ever_had_pain_or_discomfort_in_chest', 'cdq002___sp_get_it_walking_uphill_or_in_a_hurry', 'cdq003___during_an_ordinary_pace_on_level_ground', 'cdq004___if_s

In [101]:
audit_rows = []

for col in candidate_cols:
    s = df[col]
    audit_rows.append({
        "feature": col,
        "dtype": str(s.dtype),
        "missing_pct": round(s.isna().mean() * 100, 2),
        "n_unique": s.nunique(dropna=True),
        "example_values": str(get_example_values(s, n=5)),
        "is_binary_like": is_binary_series(s),
        "keep_for_model": True,
        "drop_reason": ""
    })

audit_df = pd.DataFrame(audit_rows).sort_values(
    ["missing_pct", "n_unique", "feature"],
    ascending=[False, True, True]
)

audit_df.head(100)

,feature,dtype,missing_pct,n_unique,example_values,is_binary_like,keep_for_model,drop_reason
61,mcq149___menstrual_periods_started_yet?,float64,100.00,0,[],True,True,
62,mcq151___age_in_years_at_first_menstrual_period,float64,100.00,0,[],True,True,
140,smq621___cigarettes_smoked_in_entire_life,float64,100.00,0,[],True,True,
82,mcq230c___what_kind_of_cancer_third_mention,float64,99.93,4,"[np.float64(20.0), np.float64(23.0), np.float6...",False,True,
30,cdq009g___pain_in_left_arm,float64,99.85,1,[np.float64(7.0)],False,True,
26,cdq009c___pain_in_neck,float64,99.84,1,[np.float64(3.0)],False,True,
24,cdq009a___pain_in_right_arm,float64,99.81,1,[np.float64(1.0)],True,True,
103,rhq020___age_range_at_first_menstrual_period,float64,99.68,5,"[np.float64(9.0), np.float64(3.0), np.float64(...",False,True,
106,rhq070___age_range_at_last_menstrual_period,float64,99.61,8,"[np.float64(99.0), np.float64(3.0), np.float64...",False,True,
81,mcq230b___what_kind_of_cancer_second_mention,float64,99.61,15,"[np.float64(37.0), np.float64(16.0), np.float6...",False,True,


In [102]:
MAX_MISSING_PCT = 60.0
MIN_NON_NULL = 500
MIN_VARIANCE_UNIQUE = 2

filtered_audit_df = audit_df.copy()

filtered_audit_df.loc[filtered_audit_df["missing_pct"] > MAX_MISSING_PCT, "keep_for_model"] = False
filtered_audit_df.loc[filtered_audit_df["missing_pct"] > MAX_MISSING_PCT, "drop_reason"] = "too_missing"

for idx, row in filtered_audit_df.iterrows():
    col = row["feature"]
    non_null = df[col].notna().sum()
    if non_null < MIN_NON_NULL:
        filtered_audit_df.at[idx, "keep_for_model"] = False
        filtered_audit_df.at[idx, "drop_reason"] = "too_few_non_null"
    elif row["n_unique"] < MIN_VARIANCE_UNIQUE:
        filtered_audit_df.at[idx, "keep_for_model"] = False
        filtered_audit_df.at[idx, "drop_reason"] = "constant_or_near_constant"

filtered_audit_df["keep_for_model"].value_counts()

keep_for_model
True     73
False    69
Name: count, dtype: int64

In [103]:
MANUAL_DROP_AFTER_AUDIT = [
    # Beispiele, je nach Audit:
    # "huq051___#times_receive_healthcare_over_past_year",
    # "mcq366b___doctor_told_to_increase_exercise",
    # "mcq366c___doctor_told_to_reduce_salt",
    # "mcq366d___doctor_told_to_reduce_fat_in_diet",
]

filtered_audit_df.loc[
    filtered_audit_df["feature"].isin(MANUAL_DROP_AFTER_AUDIT),
    ["keep_for_model", "drop_reason"]
] = [False, "manual_drop"]

final_features = filtered_audit_df.loc[filtered_audit_df["keep_for_model"], "feature"].tolist()

print("Final feature count:", len(final_features))
print(final_features[:80])

Final feature count: 73
['rhq305___had_both_ovaries_removed?', 'paq670___days_moderate_recreational_activities', 'rhq540___ever_use_female_hormones?', 'rhq131___ever_been_pregnant?', 'triglycerides_mg_dl', 'rhq031___had_regular_periods_in_past_12_months', 'rhq010___age_when_first_menstrual_period_occurred', 'paq625___number_of_days_moderate_work', 'cdq001___sp_ever_had_pain_or_discomfort_in_chest', 'cdq010___shortness_of_breath_on_stairs/inclines', 'alq170___past_30_days_#_times_4_5_drinks_on_an_oc', 'alq142___#_days_have_4_or_5_drinks/past_12_mos', 'alq130___avg_#_alcoholic_drinks/day___past_12_mos', 'alq151___ever_have_4/5_or_more_drinks_every_day?', 'alq121___past_12_mo_how_often_drink_alcoholic_bev', 'kiq046___leak_urine_during_nonphysical_activities', 'kiq480___how_many_times_urinate_in_night?', 'kiq044___urinated_before_reaching_the_toilet?', 'kiq042___leak_urine_during_physical_activities?', 'kiq005___how_often_have_urinary_leakage?', 'dpq040___feeling_tired_or_having_little_ene

In [104]:
final_audit_path = OUTPUT_DIR / "prediabetes_feature_audit_final.csv"
filtered_audit_df.to_csv(final_audit_path, index=False)
print("Saved:", final_audit_path.resolve())

Saved: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_prediabetes\prediabetes_feature_audit_final.csv


In [105]:
model_df = df[[TARGET] + final_features].copy()

print("Model dataframe shape:", model_df.shape)
print("Target missing:", model_df[TARGET].isna().sum())

X = model_df[final_features].copy()
y = model_df[TARGET].copy()

mask = y.notna()
X = X.loc[mask].copy()
y = y.loc[mask].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target distribution:")
print(y_train.value_counts(normalize=True).round(4))

Model dataframe shape: (7437, 74)
Target missing: 0
Train shape: (5949, 73)
Test shape: (1488, 73)
Train target distribution:
prediabetes
0    0.9082
1    0.0918
Name: proportion, dtype: float64


In [106]:
univariate_rows = []

for col in final_features:
    s = X_train[col]
    feature_type = infer_feature_type(s)

    row = {
        "feature": col,
        "feature_type": feature_type,
        "missing_pct_train": round(s.isna().mean() * 100, 2),
        "n_unique_train": s.nunique(dropna=True),
        "association_metric": None,
        "association_value": np.nan,
        "abs_association": np.nan,
        "p_value": np.nan,
        "n_train_non_null": int(s.notna().sum())
    }

    if feature_type == "categorical":
        valid = s.notna() & y_train.notna()
        if valid.sum() >= 30:
            try:
                cv = cramers_v(s[valid], y_train[valid])
                table = pd.crosstab(s[valid], y_train[valid])
                p_val = chi2_contingency(table)[1] if table.shape[0] > 1 and table.shape[1] > 1 else np.nan
                row["association_metric"] = "cramers_v"
                row["association_value"] = cv
                row["abs_association"] = abs(cv) if pd.notna(cv) else np.nan
                row["p_value"] = p_val
            except Exception:
                pass
    else:
        r, p = safe_pointbiserial(s, y_train)
        row["association_metric"] = "pointbiserial"
        row["association_value"] = r
        row["abs_association"] = abs(r) if pd.notna(r) else np.nan
        row["p_value"] = p

    univariate_rows.append(row)

univariate_df = pd.DataFrame(univariate_rows).sort_values(
    ["abs_association", "p_value"],
    ascending=[False, True]
)

univariate_df.head(50)

,feature,feature_type,missing_pct_train,n_unique_train,association_metric,association_value,abs_association,p_value,n_train_non_null
55,mcq366b___doctor_told_to_increase_exercise,categorical,0.00,3,cramers_v,0.175833,0.175833,1.150146e-40,5949
57,mcq366d___doctor_told_to_reduce_fat_in_diet,categorical,0.00,3,cramers_v,0.167165,0.167165,7.969839e-37,5949
56,mcq366c___doctor_told_to_reduce_salt,categorical,0.00,3,cramers_v,0.144598,0.144598,9.775706e-28,5949
42,slq310___usual_wake_time_on_weekdays_or_workdays,categorical,0.99,104,cramers_v,0.135698,0.135698,3.372277e-01,5890
72,age_years,numeric,0.00,48,pointbiserial,0.117479,0.117479,9.836045e-20,5949
41,slq330___usual_wake_time_on_weekends,categorical,1.13,65,cramers_v,0.115545,0.115545,1.045548e-01,5882
48,bpq020___ever_told_you_had_high_blood_pressure,categorical,0.00,3,cramers_v,0.107724,0.107724,1.021255e-15,5949
39,slq300___usual_sleep_time_on_weekdays_or_workdays,categorical,1.36,62,cramers_v,0.106239,0.106239,3.013050e-01,5868
66,"whq040___like_to_weigh_more,_less_or_same",categorical,0.00,4,cramers_v,0.102125,0.102125,2.148577e-13,5949
40,slq320___usual_sleep_time_on_weekends,categorical,1.33,53,cramers_v,0.100735,0.100735,2.196416e-01,5870


In [107]:
univariate_path = OUTPUT_DIR / "prediabetes_univariate_ranking.csv"
univariate_df.to_csv(univariate_path, index=False)
print("Saved:", univariate_path.resolve())

Saved: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_prediabetes\prediabetes_univariate_ranking.csv


In [108]:
numeric_features = []
categorical_features = []

for col in final_features:
    if infer_feature_type(X_train[col]) == "numeric":
        numeric_features.append(col)
    else:
        categorical_features.append(col)

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 10
Categorical features: 63


In [109]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

In [110]:
baseline_model = DummyClassifier(strategy="most_frequent")

baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

print("Baseline precision:", precision_score(y_test, baseline_pred, zero_division=0))
print("Baseline recall:", recall_score(y_test, baseline_pred, zero_division=0))
print("Baseline f1:", f1_score(y_test, baseline_pred, zero_division=0))

Baseline precision: 0.0
Baseline recall: 0.0
Baseline f1: 0.0


In [111]:
logreg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

In [112]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "roc_auc": "roc_auc",
    "avg_precision": "average_precision",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall"
}

logreg_cv = cross_validate(logreg_pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
rf_cv = cross_validate(rf_pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

cv_summary = pd.DataFrame({
    "model": ["logreg", "random_forest"],
    "roc_auc_mean": [logreg_cv["test_roc_auc"].mean(), rf_cv["test_roc_auc"].mean()],
    "avg_precision_mean": [logreg_cv["test_avg_precision"].mean(), rf_cv["test_avg_precision"].mean()],
    "f1_mean": [logreg_cv["test_f1"].mean(), rf_cv["test_f1"].mean()],
    "precision_mean": [logreg_cv["test_precision"].mean(), rf_cv["test_precision"].mean()],
    "recall_mean": [logreg_cv["test_recall"].mean(), rf_cv["test_recall"].mean()],
}).sort_values("avg_precision_mean", ascending=False)

cv_summary

,model,roc_auc_mean,avg_precision_mean,f1_mean,precision_mean,recall_mean
1,random_forest,0.711788,0.174493,0.057200,0.217939,0.032961
0,logreg,0.679641,0.167917,0.247861,0.159787,0.553161


In [113]:
best_pipeline = logreg_pipeline
best_model_name = "logreg"

best_pipeline.fit(X_train, y_train)

test_proba = best_pipeline.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.50).astype(int)

print("Best model:", best_model_name)
print("ROC-AUC:", round(roc_auc_score(y_test, test_proba), 4))
print("Average Precision:", round(average_precision_score(y_test, test_proba), 4))
print("Precision:", round(precision_score(y_test, test_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, test_pred, zero_division=0), 4))
print("F1:", round(f1_score(y_test, test_pred, zero_division=0), 4))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, test_pred))
print()
print(classification_report(y_test, test_pred, zero_division=0))

Best model: logreg
ROC-AUC: 0.6977
Average Precision: 0.1844
Precision: 0.1626
Recall: 0.5809
F1: 0.254

Confusion Matrix:
[[945 407]
 [ 57  79]]

              precision    recall  f1-score   support

           0       0.94      0.70      0.80      1352
           1       0.16      0.58      0.25       136

    accuracy                           0.69      1488
   macro avg       0.55      0.64      0.53      1488
weighted avg       0.87      0.69      0.75      1488



In [114]:
perm = permutation_importance(
    best_pipeline,
    X_test,
    y_test,
    n_repeats=20,
    random_state=RANDOM_STATE,
    scoring="average_precision",
    n_jobs=-1
)

perm_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

perm_df.head(30)

,feature,importance_mean,importance_std
55,mcq366b___doctor_told_to_increase_exercise,0.018912,0.009156
42,slq310___usual_wake_time_on_weekdays_or_workdays,0.007564,0.006615
57,mcq366d___doctor_told_to_reduce_fat_in_diet,0.006646,0.006201
68,slq030___how_often_do_you_snore?,0.005544,0.003937
1,paq670___days_moderate_recreational_activities,0.005272,0.002569
72,age_years,0.004701,0.005483
38,mcq160m___ever_told_you_had_thyroid_problem,0.003269,0.006093
20,dpq040___feeling_tired_or_having_little_energy,0.003217,0.004423
40,slq320___usual_sleep_time_on_weekends,0.003099,0.007359
12,alq130___avg_#_alcoholic_drinks/day___past_12_mos,0.002891,0.002925


# Prediabetes-focused only run
This section restricts the feature space to prediabetes-relevant and quiz-eligible variables only.

### Purpose
Compare a broad questionnaire model (Set 1) against a prediabetes-focused model (Set 2).

### Principle
We intentionally exclude:
- direct target / diabetes leakage
- doctor-feedback / post-hoc intervention variables
- healthcare utilization proxies
- broad morbidity proxies that are not practical for a quiz

We keep only:
- plausible symptom proxies
- lifestyle / activity signals
- sleep / circadian signals
- family risk / related cardiometabolic context
- a few subjective weight / fatigue signals

In [126]:
PREDIABETES_FOCUSED_FEATURES = [
    # Urination / possible glucose-related symptom proxies
    "kiq480___how_many_times_urinate_in_night?",
    "kiq044___urinated_before_reaching_the_toilet?",
    "kiq046___leak_urine_during_nonphysical_activities",
    "kiq005___how_often_have_urinary_leakage?",

    # Fatigue / subjective symptom burden
    "dpq040___feeling_tired_or_having_little_energy",

    # Activity / sedentary risk proxies
    "paq650___vigorous_recreational_activities",
    "paq665___moderate_recreational_activities",
    "paq670___days_moderate_recreational_activities",
    "paq605___vigorous_work_activity",
    "paq625___number_of_days_moderate_work",

    # Sleep / circadian pattern
    "slq030___how_often_do_you_snore?",
    "sleep_hours_weekday",
    "sleep_hours_weekend",
    "social_jetlag",
    "sin_weekday_bedtime",
    "cos_weekday_bedtime",
    "sin_weekday_wake",
    "cos_weekday_wake",
    

    # Weight / body perception proxy
    "whq040___like_to_weigh_more,_less_or_same",

    # Related cardiometabolic context
    "bpq020___ever_told_you_had_high_blood_pressure",
    "mcq300c___close_relative_had_diabetes",

    # Lifestyle / alcohol
    "alq111___ever_had_a_drink_of_any_kind_of_alcohol",
    "alq121___past_12_mo_how_often_drink_alcoholic_bev",
    "alq130___avg_#_alcoholic_drinks/day___past_12_mos",
    "alq142___#_days_have_4_or_5_drinks/past_12_mos",
    "alq170___past_30_days_#_times_4_5_drinks_on_an_oc",

    # Age & Gender
    "age_years",
    "gender"

    # Meds & Labs
    "med_count",
    "fasting_glucose_mg_dl"
]

prediabetes_focused_available = [c for c in PREDIABETES_FOCUSED_FEATURES if c in df.columns]
prediabetes_focused_missing = [c for c in PREDIABETES_FOCUSED_FEATURES if c not in df.columns]

print("Available prediabetes-focused features:", len(prediabetes_focused_available))
print(prediabetes_focused_available)
print()
print("Missing requested features:", len(prediabetes_focused_missing))
print(prediabetes_focused_missing)

Available prediabetes-focused features: 28
['kiq480___how_many_times_urinate_in_night?', 'kiq044___urinated_before_reaching_the_toilet?', 'kiq046___leak_urine_during_nonphysical_activities', 'kiq005___how_often_have_urinary_leakage?', 'dpq040___feeling_tired_or_having_little_energy', 'paq650___vigorous_recreational_activities', 'paq665___moderate_recreational_activities', 'paq670___days_moderate_recreational_activities', 'paq605___vigorous_work_activity', 'paq625___number_of_days_moderate_work', 'slq030___how_often_do_you_snore?', 'sleep_hours_weekday', 'sleep_hours_weekend', 'social_jetlag', 'sin_weekday_bedtime', 'cos_weekday_bedtime', 'sin_weekday_wake', 'cos_weekday_wake', 'whq040___like_to_weigh_more,_less_or_same', 'bpq020___ever_told_you_had_high_blood_pressure', 'mcq300c___close_relative_had_diabetes', 'alq111___ever_had_a_drink_of_any_kind_of_alcohol', 'alq121___past_12_mo_how_often_drink_alcoholic_bev', 'alq130___avg_#_alcoholic_drinks/day___past_12_mos', 'alq142___#_days_hav

In [127]:
prediabetes_focus_audit_rows = []

for col in prediabetes_focused_available:
    s = df[col]
    prediabetes_focus_audit_rows.append({
        "feature": col,
        "dtype": str(s.dtype),
        "missing_pct": round(s.isna().mean() * 100, 2),
        "n_unique": s.nunique(dropna=True),
        "example_values": str(get_example_values(s, n=5)),
        "is_binary_like": is_binary_series(s),
    })

prediabetes_focus_audit_df = pd.DataFrame(prediabetes_focus_audit_rows).sort_values(
    ["missing_pct", "n_unique", "feature"],
    ascending=[False, True, True]
)

prediabetes_focus_audit_df

,feature,dtype,missing_pct,n_unique,example_values,is_binary_like
7,paq670___days_moderate_recreational_activities,float64,57.70,8,"[np.float64(4.0), np.float64(5.0), np.float64(...",False
27,fasting_glucose_mg_dl,float64,56.56,219,"[np.float64(103.0), np.float64(92.0), np.float...",False
9,paq625___number_of_days_moderate_work,float64,54.20,8,"[np.float64(5.0), np.float64(2.0), np.float64(...",False
25,alq170___past_30_days_#_times_4_5_drinks_on_an_oc,float64,35.35,23,"[np.float64(0.0), np.float64(30.0), np.float64...",False
24,alq142___#_days_have_4_or_5_drinks/past_12_mos,float64,34.89,13,"[np.float64(0.0), np.float64(4.0), np.float64(...",False
23,alq130___avg_#_alcoholic_drinks/day___past_12_mos,float64,34.89,15,"[np.float64(1.0), np.float64(6.0), np.float64(...",False
22,alq121___past_12_mo_how_often_drink_alcoholic_bev,float64,21.68,13,"[np.float64(10.0), np.float64(0.0), np.float64...",False
2,kiq046___leak_urine_during_nonphysical_activities,float64,18.14,4,"[np.float64(2.0), np.float64(1.0), np.float64(...",False
0,kiq480___how_many_times_urinate_in_night?,float64,18.14,8,"[np.float64(0.0), np.float64(2.0), np.float64(...",False
1,kiq044___urinated_before_reaching_the_toilet?,float64,18.13,4,"[np.float64(2.0), np.float64(1.0), np.float64(...",False


In [128]:
PREDIABETES_FOCUS_MANUAL_DROP = [
    "alq111___ever_had_a_drink_of_any_kind_of_alcohol",
    "alq121___past_12_mo_how_often_drink_alcoholic_bev",
    "alq142___#_days_have_4_or_5_drinks/past_12_mos",
    "kiq005___how_often_have_urinary_leakage",
    "kiq044___urinated_before_reaching_the_toilet",
    "kiq046___leak_urine_during_nonphysical_activities",
]

PREDIABETES_FOCUS_MAX_MISSING_PCT = 60.0

prediabetes_focused_final_features = [
    c for c in prediabetes_focused_available
    if c not in PREDIABETES_FOCUS_MANUAL_DROP
    and df[c].isna().mean() * 100 <= PREDIABETES_FOCUS_MAX_MISSING_PCT
]

print("Final prediabetes-focused features:", len(prediabetes_focused_final_features))
print(prediabetes_focused_final_features)

Final prediabetes-focused features: 24
['kiq480___how_many_times_urinate_in_night?', 'kiq044___urinated_before_reaching_the_toilet?', 'kiq005___how_often_have_urinary_leakage?', 'dpq040___feeling_tired_or_having_little_energy', 'paq650___vigorous_recreational_activities', 'paq665___moderate_recreational_activities', 'paq670___days_moderate_recreational_activities', 'paq605___vigorous_work_activity', 'paq625___number_of_days_moderate_work', 'slq030___how_often_do_you_snore?', 'sleep_hours_weekday', 'sleep_hours_weekend', 'social_jetlag', 'sin_weekday_bedtime', 'cos_weekday_bedtime', 'sin_weekday_wake', 'cos_weekday_wake', 'whq040___like_to_weigh_more,_less_or_same', 'bpq020___ever_told_you_had_high_blood_pressure', 'mcq300c___close_relative_had_diabetes', 'alq130___avg_#_alcoholic_drinks/day___past_12_mos', 'alq170___past_30_days_#_times_4_5_drinks_on_an_oc', 'age_years', 'fasting_glucose_mg_dl']


In [129]:
prediabetes_focus_df = df[[TARGET] + prediabetes_focused_final_features].copy()

mask = prediabetes_focus_df[TARGET].notna()
X_pf = prediabetes_focus_df.loc[mask, prediabetes_focused_final_features].copy()
y_pf = prediabetes_focus_df.loc[mask, TARGET].astype(int).copy()

X_train_pf, X_test_pf, y_train_pf, y_test_pf = train_test_split(
    X_pf, y_pf,
    test_size=TEST_SIZE,
    stratify=y_pf,
    random_state=RANDOM_STATE
)

print("Prediabetes-focused train shape:", X_train_pf.shape)
print("Prediabetes-focused test shape:", X_test_pf.shape)
print("Train target distribution:")
print(y_train_pf.value_counts(normalize=True).round(4))

Prediabetes-focused train shape: (5949, 24)
Prediabetes-focused test shape: (1488, 24)
Train target distribution:
prediabetes
0    0.9082
1    0.0918
Name: proportion, dtype: float64


In [130]:
pf_univariate_rows = []

for col in prediabetes_focused_final_features:
    s = X_train_pf[col]
    feature_type = infer_feature_type(s)

    row = {
        "feature": col,
        "feature_type": feature_type,
        "missing_pct_train": round(s.isna().mean() * 100, 2),
        "n_unique_train": s.nunique(dropna=True),
        "association_metric": None,
        "association_value": np.nan,
        "abs_association": np.nan,
        "p_value": np.nan,
        "n_train_non_null": int(s.notna().sum())
    }

    if feature_type == "categorical":
        valid = s.notna() & y_train_pf.notna()
        if valid.sum() >= 30:
            try:
                cv = cramers_v(s[valid], y_train_pf[valid])
                table = pd.crosstab(s[valid], y_train_pf[valid])
                p_val = chi2_contingency(table)[1] if table.shape[0] > 1 and table.shape[1] > 1 else np.nan
                row["association_metric"] = "cramers_v"
                row["association_value"] = cv
                row["abs_association"] = abs(cv) if pd.notna(cv) else np.nan
                row["p_value"] = p_val
            except Exception:
                pass
    else:
        r, p = safe_pointbiserial(s, y_train_pf)
        row["association_metric"] = "pointbiserial"
        row["association_value"] = r
        row["abs_association"] = abs(r) if pd.notna(r) else np.nan
        row["p_value"] = p

    pf_univariate_rows.append(row)

pf_univariate_df = pd.DataFrame(pf_univariate_rows).sort_values(
    ["abs_association", "p_value"],
    ascending=[False, True]
)

pf_univariate_df

,feature,feature_type,missing_pct_train,n_unique_train,association_metric,association_value,abs_association,p_value,n_train_non_null
22,age_years,numeric,0.00,48,pointbiserial,0.117479,0.117479,9.836045e-20,5949
18,bpq020___ever_told_you_had_high_blood_pressure,categorical,0.00,3,cramers_v,0.107724,0.107724,1.021255e-15,5949
17,"whq040___like_to_weigh_more,_less_or_same",categorical,0.00,4,cramers_v,0.102125,0.102125,2.148577e-13,5949
9,slq030___how_often_do_you_snore?,categorical,0.00,6,cramers_v,0.091155,0.091155,1.811455e-09,5949
19,mcq300c___close_relative_had_diabetes,categorical,6.35,3,cramers_v,0.079418,0.079418,2.343901e-08,5571
6,paq670___days_moderate_recreational_activities,categorical,57.61,8,cramers_v,0.062153,0.062153,2.036523e-01,2522
2,kiq005___how_often_have_urinary_leakage?,categorical,18.10,7,cramers_v,0.061439,0.061439,5.326282e-03,4872
3,dpq040___feeling_tired_or_having_little_energy,categorical,12.93,6,cramers_v,0.061242,0.061242,1.599212e-03,5180
1,kiq044___urinated_before_reaching_the_toilet?,categorical,18.14,4,cramers_v,0.051428,0.051428,4.902608e-03,4870
0,kiq480___how_many_times_urinate_in_night?,categorical,18.15,8,cramers_v,0.044273,0.044273,2.159261e-01,4869


In [131]:
numeric_features_pf = []
categorical_features_pf = []

for col in prediabetes_focused_final_features:
    if infer_feature_type(X_train_pf[col]) == "numeric":
        numeric_features_pf.append(col)
    else:
        categorical_features_pf.append(col)

print("Prediabetes-focused numeric features:", numeric_features_pf)
print()
print("Prediabetes-focused categorical features:", categorical_features_pf)

Prediabetes-focused numeric features: ['sleep_hours_weekday', 'sleep_hours_weekend', 'social_jetlag', 'sin_weekday_bedtime', 'cos_weekday_bedtime', 'sin_weekday_wake', 'cos_weekday_wake', 'alq130___avg_#_alcoholic_drinks/day___past_12_mos', 'alq170___past_30_days_#_times_4_5_drinks_on_an_oc', 'age_years', 'fasting_glucose_mg_dl']

Prediabetes-focused categorical features: ['kiq480___how_many_times_urinate_in_night?', 'kiq044___urinated_before_reaching_the_toilet?', 'kiq005___how_often_have_urinary_leakage?', 'dpq040___feeling_tired_or_having_little_energy', 'paq650___vigorous_recreational_activities', 'paq665___moderate_recreational_activities', 'paq670___days_moderate_recreational_activities', 'paq605___vigorous_work_activity', 'paq625___number_of_days_moderate_work', 'slq030___how_often_do_you_snore?', 'whq040___like_to_weigh_more,_less_or_same', 'bpq020___ever_told_you_had_high_blood_pressure', 'mcq300c___close_relative_had_diabetes']


In [132]:
preprocessor_pf = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_features_pf),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features_pf),
    ],
    remainder="drop"
)

In [133]:
logreg_pipeline_pf = Pipeline(steps=[
    ("preprocessor", preprocessor_pf),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

rf_pipeline_pf = Pipeline(steps=[
    ("preprocessor", preprocessor_pf),
    ("model", RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

In [134]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "roc_auc": "roc_auc",
    "avg_precision": "average_precision",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall"
}

logreg_cv_pf = cross_validate(logreg_pipeline_pf, X_train_pf, y_train_pf, cv=cv, scoring=scoring, n_jobs=-1)
rf_cv_pf = cross_validate(rf_pipeline_pf, X_train_pf, y_train_pf, cv=cv, scoring=scoring, n_jobs=-1)

cv_summary_pf = pd.DataFrame({
    "model": ["logreg_prediabetes_focused", "random_forest_prediabetes_focused"],
    "roc_auc_mean": [logreg_cv_pf["test_roc_auc"].mean(), rf_cv_pf["test_roc_auc"].mean()],
    "avg_precision_mean": [logreg_cv_pf["test_avg_precision"].mean(), rf_cv_pf["test_avg_precision"].mean()],
    "f1_mean": [logreg_cv_pf["test_f1"].mean(), rf_cv_pf["test_f1"].mean()],
    "precision_mean": [logreg_cv_pf["test_precision"].mean(), rf_cv_pf["test_precision"].mean()],
    "recall_mean": [logreg_cv_pf["test_recall"].mean(), rf_cv_pf["test_recall"].mean()],
}).sort_values("avg_precision_mean", ascending=False)

cv_summary_pf

,model,roc_auc_mean,avg_precision_mean,f1_mean,precision_mean,recall_mean
1,random_forest_prediabetes_focused,0.674702,0.156912,0.043574,0.161962,0.025655
0,logreg_prediabetes_focused,0.666762,0.151010,0.231264,0.142142,0.621001


In [135]:
# Choose the practically better model after inspection of cv_summary_pf.
# Defaulting to logistic regression as a robust starting point for imbalanced questionnaire data.

best_pipeline_pf = logreg_pipeline_pf
best_model_name_pf = "logreg_prediabetes_focused"

best_pipeline_pf.fit(X_train_pf, y_train_pf)

test_proba_pf = best_pipeline_pf.predict_proba(X_test_pf)[:, 1]
test_pred_pf = (test_proba_pf >= 0.50).astype(int)

print("Best prediabetes-focused model:", best_model_name_pf)
print("ROC-AUC:", round(roc_auc_score(y_test_pf, test_proba_pf), 4))
print("Average Precision:", round(average_precision_score(y_test_pf, test_proba_pf), 4))
print("Precision:", round(precision_score(y_test_pf, test_pred_pf, zero_division=0), 4))
print("Recall:", round(recall_score(y_test_pf, test_pred_pf, zero_division=0), 4))
print("F1:", round(f1_score(y_test_pf, test_pred_pf, zero_division=0), 4))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test_pf, test_pred_pf))
print()
print(classification_report(y_test_pf, test_pred_pf, zero_division=0))

Best prediabetes-focused model: logreg_prediabetes_focused
ROC-AUC: 0.6668
Average Precision: 0.1601
Precision: 0.1433
Recall: 0.6544
F1: 0.2351

Confusion Matrix:
[[820 532]
 [ 47  89]]

              precision    recall  f1-score   support

           0       0.95      0.61      0.74      1352
           1       0.14      0.65      0.24       136

    accuracy                           0.61      1488
   macro avg       0.54      0.63      0.49      1488
weighted avg       0.87      0.61      0.69      1488



In [136]:
perm_pf = permutation_importance(
    best_pipeline_pf,
    X_test_pf,
    y_test_pf,
    n_repeats=20,
    random_state=RANDOM_STATE,
    scoring="average_precision",
    n_jobs=-1
)

perm_df_pf = pd.DataFrame({
    "feature": X_test_pf.columns,
    "importance_mean": perm_pf.importances_mean,
    "importance_std": perm_pf.importances_std
}).sort_values("importance_mean", ascending=False)

perm_df_pf

,feature,importance_mean,importance_std
14,cos_weekday_bedtime,0.016089,0.003325
11,sleep_hours_weekend,0.012824,0.004622
22,age_years,0.012353,0.007608
9,slq030___how_often_do_you_snore?,0.011991,0.005513
19,mcq300c___close_relative_had_diabetes,0.010296,0.007520
3,dpq040___feeling_tired_or_having_little_energy,0.010090,0.006997
17,"whq040___like_to_weigh_more,_less_or_same",0.006937,0.009648
8,paq625___number_of_days_moderate_work,0.006556,0.005115
0,kiq480___how_many_times_urinate_in_night?,0.006350,0.002289
5,paq665___moderate_recreational_activities,0.005332,0.004768


In [137]:
combined_df_pf = (
    pf_univariate_df.merge(perm_df_pf, on="feature", how="left")
    .merge(prediabetes_focus_audit_df[["feature", "missing_pct", "n_unique"]], on="feature", how="left")
)

combined_df_pf = combined_df_pf.sort_values(
    ["importance_mean", "abs_association"],
    ascending=[False, False]
)

combined_df_pf

,feature,feature_type,missing_pct_train,n_unique_train,association_metric,association_value,abs_association,p_value,n_train_non_null,importance_mean,importance_std,missing_pct,n_unique
21,cos_weekday_bedtime,numeric,1.36,58,pointbiserial,0.012238,0.012238,3.485854e-01,5868,0.016089,0.003325,1.29,59
17,sleep_hours_weekend,numeric,0.91,24,pointbiserial,-0.016076,0.016076,2.171741e-01,5895,0.012824,0.004622,0.86,24
0,age_years,numeric,0.00,48,pointbiserial,0.117479,0.117479,9.836045e-20,5949,0.012353,0.007608,0.00,48
3,slq030___how_often_do_you_snore?,categorical,0.00,6,cramers_v,0.091155,0.091155,1.811455e-09,5949,0.011991,0.005513,0.00,6
4,mcq300c___close_relative_had_diabetes,categorical,6.35,3,cramers_v,0.079418,0.079418,2.343901e-08,5571,0.010296,0.007520,6.20,3
7,dpq040___feeling_tired_or_having_little_energy,categorical,12.93,6,cramers_v,0.061242,0.061242,1.599212e-03,5180,0.010090,0.006997,13.03,6
2,"whq040___like_to_weigh_more,_less_or_same",categorical,0.00,4,cramers_v,0.102125,0.102125,2.148577e-13,5949,0.006937,0.009648,0.00,4
10,paq625___number_of_days_moderate_work,categorical,54.23,8,cramers_v,0.042735,0.042735,6.632526e-01,2723,0.006556,0.005115,54.20,8
9,kiq480___how_many_times_urinate_in_night?,categorical,18.15,8,cramers_v,0.044273,0.044273,2.159261e-01,4869,0.006350,0.002289,18.14,8
14,paq665___moderate_recreational_activities,categorical,0.00,2,cramers_v,0.021243,0.021243,1.013299e-01,5949,0.005332,0.004768,0.00,2


## Threshold Tuning

In [138]:
threshold_rows = []

for t in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    pred_t = (test_proba_pf >= t).astype(int)
    threshold_rows.append({
        "threshold": t,
        "precision": precision_score(y_test_pf, pred_t, zero_division=0),
        "recall": recall_score(y_test_pf, pred_t, zero_division=0),
        "f1": f1_score(y_test_pf, pred_t, zero_division=0),
        "positives_predicted": int(pred_t.sum())
    })

threshold_df_pf = pd.DataFrame(threshold_rows).sort_values("f1", ascending=False)
threshold_df_pf

,threshold,precision,recall,f1,positives_predicted
6,0.50,0.143317,0.654412,0.235139,621
5,0.40,0.123025,0.801471,0.213307,886
4,0.35,0.119478,0.875000,0.210247,996
3,0.30,0.110314,0.904412,0.196643,1115
2,0.25,0.103533,0.926471,0.186253,1217
1,0.20,0.098248,0.948529,0.178054,1313
0,0.15,0.095890,0.977941,0.174655,1387


# Final 6

In [139]:
# Final compact quiz run for prediabetes
# Goal: test a very small, product-friendly feature set

PREDIABETES_FINAL_QUIZ_FEATURES = [
    "cos_weekday_bedtime",
    "sleep_hours_weekend",
    "slq030___how_often_do_you_snore?",
    "mcq300c___close_relative_had_diabetes",
    "paq650___vigorous_recreational_activities",
    "whq040___like_to_weigh_more,_less_or_same",
]

final_quiz_available = [c for c in PREDIABETES_FINAL_QUIZ_FEATURES if c in df.columns]
final_quiz_missing = [c for c in PREDIABETES_FINAL_QUIZ_FEATURES if c not in df.columns]

print("Available final quiz features:", len(final_quiz_available))
print(final_quiz_available)
print()
print("Missing requested features:", len(final_quiz_missing))
print(final_quiz_missing)

Available final quiz features: 6
['cos_weekday_bedtime', 'sleep_hours_weekend', 'slq030___how_often_do_you_snore?', 'mcq300c___close_relative_had_diabetes', 'paq650___vigorous_recreational_activities', 'whq040___like_to_weigh_more,_less_or_same']

Missing requested features: 0
[]


In [140]:
final_quiz_audit_rows = []

for col in final_quiz_available:
    s = df[col]
    final_quiz_audit_rows.append({
        "feature": col,
        "dtype": str(s.dtype),
        "missing_pct": round(s.isna().mean() * 100, 2),
        "n_unique": s.nunique(dropna=True),
        "example_values": str(get_example_values(s, n=10)),
        "is_binary_like": is_binary_series(s),
    })

final_quiz_audit_df = pd.DataFrame(final_quiz_audit_rows).sort_values(
    ["missing_pct", "n_unique", "feature"],
    ascending=[False, True, True]
)

final_quiz_audit_df

,feature,dtype,missing_pct,n_unique,example_values,is_binary_like
3,mcq300c___close_relative_had_diabetes,float64,6.20,3,"[np.float64(1.0), np.float64(2.0), np.float64(...",False
0,cos_weekday_bedtime,float64,1.29,59,"[np.float64(0.8660254037844384), np.float64(1....",False
1,sleep_hours_weekend,float64,0.86,24,"[np.float64(8.0), np.float64(13.0), np.float64...",False
4,paq650___vigorous_recreational_activities,float64,0.00,2,"[np.float64(1.0), np.float64(2.0)]",False
5,"whq040___like_to_weigh_more,_less_or_same",float64,0.00,4,"[np.float64(2.0), np.float64(1.0), np.float64(...",False
2,slq030___how_often_do_you_snore?,float64,0.00,6,"[np.float64(1.0), np.float64(0.0), np.float64(...",False


In [141]:
final_quiz_df = df[[TARGET] + final_quiz_available].copy()

mask = final_quiz_df[TARGET].notna()
X_fq = final_quiz_df.loc[mask, final_quiz_available].copy()
y_fq = final_quiz_df.loc[mask, TARGET].astype(int).copy()

X_train_fq, X_test_fq, y_train_fq, y_test_fq = train_test_split(
    X_fq, y_fq,
    test_size=TEST_SIZE,
    stratify=y_fq,
    random_state=RANDOM_STATE
)

print("Final quiz train shape:", X_train_fq.shape)
print("Final quiz test shape:", X_test_fq.shape)
print("Train target distribution:")
print(y_train_fq.value_counts(normalize=True).round(4))

Final quiz train shape: (5949, 6)
Final quiz test shape: (1488, 6)
Train target distribution:
prediabetes
0    0.9082
1    0.0918
Name: proportion, dtype: float64


In [142]:
fq_univariate_rows = []

for col in final_quiz_available:
    s = X_train_fq[col]
    feature_type = infer_feature_type(s)

    row = {
        "feature": col,
        "feature_type": feature_type,
        "missing_pct_train": round(s.isna().mean() * 100, 2),
        "n_unique_train": s.nunique(dropna=True),
        "association_metric": None,
        "association_value": np.nan,
        "abs_association": np.nan,
        "p_value": np.nan,
        "n_train_non_null": int(s.notna().sum())
    }

    if feature_type == "categorical":
        valid = s.notna() & y_train_fq.notna()
        if valid.sum() >= 30:
            try:
                cv = cramers_v(s[valid], y_train_fq[valid])
                table = pd.crosstab(s[valid], y_train_fq[valid])
                p_val = chi2_contingency(table)[1] if table.shape[0] > 1 and table.shape[1] > 1 else np.nan
                row["association_metric"] = "cramers_v"
                row["association_value"] = cv
                row["abs_association"] = abs(cv) if pd.notna(cv) else np.nan
                row["p_value"] = p_val
            except Exception:
                pass
    else:
        r, p = safe_pointbiserial(s, y_train_fq)
        row["association_metric"] = "pointbiserial"
        row["association_value"] = r
        row["abs_association"] = abs(r) if pd.notna(r) else np.nan
        row["p_value"] = p

    fq_univariate_rows.append(row)

fq_univariate_df = pd.DataFrame(fq_univariate_rows).sort_values(
    ["abs_association", "p_value"],
    ascending=[False, True]
)

fq_univariate_df

,feature,feature_type,missing_pct_train,n_unique_train,association_metric,association_value,abs_association,p_value,n_train_non_null
5,"whq040___like_to_weigh_more,_less_or_same",categorical,0.00,4,cramers_v,0.102125,0.102125,2.148577e-13,5949
2,slq030___how_often_do_you_snore?,categorical,0.00,6,cramers_v,0.091155,0.091155,1.811455e-09,5949
3,mcq300c___close_relative_had_diabetes,categorical,6.35,3,cramers_v,0.079418,0.079418,2.343901e-08,5571
4,paq650___vigorous_recreational_activities,categorical,0.00,2,cramers_v,0.039632,0.039632,2.236881e-03,5949
1,sleep_hours_weekend,numeric,0.91,24,pointbiserial,-0.016076,0.016076,2.171741e-01,5895
0,cos_weekday_bedtime,numeric,1.36,58,pointbiserial,0.012238,0.012238,3.485854e-01,5868


In [143]:
numeric_features_fq = []
categorical_features_fq = []

for col in final_quiz_available:
    if infer_feature_type(X_train_fq[col]) == "numeric":
        numeric_features_fq.append(col)
    else:
        categorical_features_fq.append(col)

print("Final quiz numeric features:", numeric_features_fq)
print()
print("Final quiz categorical features:", categorical_features_fq)

Final quiz numeric features: ['cos_weekday_bedtime', 'sleep_hours_weekend']

Final quiz categorical features: ['slq030___how_often_do_you_snore?', 'mcq300c___close_relative_had_diabetes', 'paq650___vigorous_recreational_activities', 'whq040___like_to_weigh_more,_less_or_same']


In [144]:
preprocessor_fq = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), numeric_features_fq),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features_fq),
    ],
    remainder="drop"
)

In [145]:
baseline_model_fq = DummyClassifier(strategy="most_frequent")

baseline_model_fq.fit(X_train_fq, y_train_fq)
baseline_pred_fq = baseline_model_fq.predict(X_test_fq)

print("Baseline precision:", precision_score(y_test_fq, baseline_pred_fq, zero_division=0))
print("Baseline recall:", recall_score(y_test_fq, baseline_pred_fq, zero_division=0))
print("Baseline f1:", f1_score(y_test_fq, baseline_pred_fq, zero_division=0))

Baseline precision: 0.0
Baseline recall: 0.0
Baseline f1: 0.0


In [146]:
logreg_pipeline_fq = Pipeline(steps=[
    ("preprocessor", preprocessor_fq),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

rf_pipeline_fq = Pipeline(steps=[
    ("preprocessor", preprocessor_fq),
    ("model", RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

In [147]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "roc_auc": "roc_auc",
    "avg_precision": "average_precision",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall"
}

logreg_cv_fq = cross_validate(logreg_pipeline_fq, X_train_fq, y_train_fq, cv=cv, scoring=scoring, n_jobs=-1)
rf_cv_fq = cross_validate(rf_pipeline_fq, X_train_fq, y_train_fq, cv=cv, scoring=scoring, n_jobs=-1)

cv_summary_fq = pd.DataFrame({
    "model": ["logreg_final_quiz", "random_forest_final_quiz"],
    "roc_auc_mean": [logreg_cv_fq["test_roc_auc"].mean(), rf_cv_fq["test_roc_auc"].mean()],
    "avg_precision_mean": [logreg_cv_fq["test_avg_precision"].mean(), rf_cv_fq["test_avg_precision"].mean()],
    "f1_mean": [logreg_cv_fq["test_f1"].mean(), rf_cv_fq["test_f1"].mean()],
    "precision_mean": [logreg_cv_fq["test_precision"].mean(), rf_cv_fq["test_precision"].mean()],
    "recall_mean": [logreg_cv_fq["test_recall"].mean(), rf_cv_fq["test_recall"].mean()],
}).sort_values("avg_precision_mean", ascending=False)

cv_summary_fq

,model,roc_auc_mean,avg_precision_mean,f1_mean,precision_mean,recall_mean
1,random_forest_final_quiz,0.627799,0.134600,0.207203,0.148374,0.344287
0,logreg_final_quiz,0.636517,0.133748,0.217063,0.130597,0.643003


In [148]:
# Default to logistic regression unless RF clearly wins in practical metrics
best_pipeline_fq = logreg_pipeline_fq
best_model_name_fq = "logreg_final_quiz"

best_pipeline_fq.fit(X_train_fq, y_train_fq)

test_proba_fq = best_pipeline_fq.predict_proba(X_test_fq)[:, 1]
test_pred_fq = (test_proba_fq >= 0.50).astype(int)

print("Best final quiz model:", best_model_name_fq)
print("ROC-AUC:", round(roc_auc_score(y_test_fq, test_proba_fq), 4))
print("Average Precision:", round(average_precision_score(y_test_fq, test_proba_fq), 4))
print("Precision:", round(precision_score(y_test_fq, test_pred_fq, zero_division=0), 4))
print("Recall:", round(recall_score(y_test_fq, test_pred_fq, zero_division=0), 4))
print("F1:", round(f1_score(y_test_fq, test_pred_fq, zero_division=0), 4))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test_fq, test_pred_fq))
print()
print(classification_report(y_test_fq, test_pred_fq, zero_division=0))

Best final quiz model: logreg_final_quiz
ROC-AUC: 0.6543
Average Precision: 0.1474
Precision: 0.1277
Recall: 0.6618
F1: 0.214

Confusion Matrix:
[[737 615]
 [ 46  90]]

              precision    recall  f1-score   support

           0       0.94      0.55      0.69      1352
           1       0.13      0.66      0.21       136

    accuracy                           0.56      1488
   macro avg       0.53      0.60      0.45      1488
weighted avg       0.87      0.56      0.65      1488



In [149]:
perm_fq = permutation_importance(
    best_pipeline_fq,
    X_test_fq,
    y_test_fq,
    n_repeats=20,
    random_state=RANDOM_STATE,
    scoring="average_precision",
    n_jobs=-1
)

perm_df_fq = pd.DataFrame({
    "feature": X_test_fq.columns,
    "importance_mean": perm_fq.importances_mean,
    "importance_std": perm_fq.importances_std
}).sort_values("importance_mean", ascending=False)

perm_df_fq

,feature,importance_mean,importance_std
5,"whq040___like_to_weigh_more,_less_or_same",0.016255,0.007604
3,mcq300c___close_relative_had_diabetes,0.012248,0.012377
2,slq030___how_often_do_you_snore?,0.008529,0.009650
4,paq650___vigorous_recreational_activities,0.007429,0.003629
0,cos_weekday_bedtime,0.002749,0.003485
1,sleep_hours_weekend,-0.011010,0.006168


In [150]:
combined_df_fq = (
    fq_univariate_df.merge(perm_df_fq, on="feature", how="left")
    .merge(final_quiz_audit_df[["feature", "missing_pct", "n_unique"]], on="feature", how="left")
)

combined_df_fq = combined_df_fq.sort_values(
    ["importance_mean", "abs_association"],
    ascending=[False, False]
)

combined_df_fq

,feature,feature_type,missing_pct_train,n_unique_train,association_metric,association_value,abs_association,p_value,n_train_non_null,importance_mean,importance_std,missing_pct,n_unique
0,"whq040___like_to_weigh_more,_less_or_same",categorical,0.00,4,cramers_v,0.102125,0.102125,2.148577e-13,5949,0.016255,0.007604,0.00,4
2,mcq300c___close_relative_had_diabetes,categorical,6.35,3,cramers_v,0.079418,0.079418,2.343901e-08,5571,0.012248,0.012377,6.20,3
1,slq030___how_often_do_you_snore?,categorical,0.00,6,cramers_v,0.091155,0.091155,1.811455e-09,5949,0.008529,0.009650,0.00,6
3,paq650___vigorous_recreational_activities,categorical,0.00,2,cramers_v,0.039632,0.039632,2.236881e-03,5949,0.007429,0.003629,0.00,2
5,cos_weekday_bedtime,numeric,1.36,58,pointbiserial,0.012238,0.012238,3.485854e-01,5868,0.002749,0.003485,1.29,59
4,sleep_hours_weekend,numeric,0.91,24,pointbiserial,-0.016076,0.016076,2.171741e-01,5895,-0.011010,0.006168,0.86,24


In [151]:
comparison_test_fq = pd.DataFrame([
    {
        "run": "set1_all_questionnaire",
        "model": best_model_name,
        "roc_auc": roc_auc_score(y_test, test_proba),
        "avg_precision": average_precision_score(y_test, test_proba),
        "precision": precision_score(y_test, test_pred, zero_division=0),
        "recall": recall_score(y_test, test_pred, zero_division=0),
        "f1": f1_score(y_test, test_pred, zero_division=0),
        "n_features": len(final_features),
    },
    {
        "run": "set2_prediabetes_focused",
        "model": best_model_name_pf,
        "roc_auc": roc_auc_score(y_test_pf, test_proba_pf),
        "avg_precision": average_precision_score(y_test_pf, test_proba_pf),
        "precision": precision_score(y_test_pf, test_pred_pf, zero_division=0),
        "recall": recall_score(y_test_pf, test_pred_pf, zero_division=0),
        "f1": f1_score(y_test_pf, test_pred_pf, zero_division=0),
        "n_features": len(prediabetes_focused_final_features),
    },
    {
        "run": "set3_final_quiz_6_features",
        "model": best_model_name_fq,
        "roc_auc": roc_auc_score(y_test_fq, test_proba_fq),
        "avg_precision": average_precision_score(y_test_fq, test_proba_fq),
        "precision": precision_score(y_test_fq, test_pred_fq, zero_division=0),
        "recall": recall_score(y_test_fq, test_pred_fq, zero_division=0),
        "f1": f1_score(y_test_fq, test_pred_fq, zero_division=0),
        "n_features": len(final_quiz_available),
    }
])

comparison_test_fq.sort_values("avg_precision", ascending=False)

,run,model,roc_auc,avg_precision,precision,recall,f1,n_features
0,set1_all_questionnaire,logreg,0.697686,0.184407,0.162551,0.580882,0.254019,73
1,set2_prediabetes_focused,logreg_prediabetes_focused,0.666763,0.160069,0.143317,0.654412,0.235139,24
2,set3_final_quiz_6_features,logreg_final_quiz,0.654298,0.147373,0.127660,0.661765,0.214031,6


In [152]:
threshold_rows_fq = []

for t in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    pred_t = (test_proba_fq >= t).astype(int)
    threshold_rows_fq.append({
        "threshold": t,
        "precision": precision_score(y_test_fq, pred_t, zero_division=0),
        "recall": recall_score(y_test_fq, pred_t, zero_division=0),
        "f1": f1_score(y_test_fq, pred_t, zero_division=0),
        "positives_predicted": int(pred_t.sum())
    })

threshold_df_fq = pd.DataFrame(threshold_rows_fq).sort_values("f1", ascending=False)
threshold_df_fq

,threshold,precision,recall,f1,positives_predicted
6,0.50,0.127660,0.661765,0.214031,705
5,0.40,0.113503,0.852941,0.200345,1022
4,0.35,0.108062,0.926471,0.193548,1166
3,0.30,0.102482,0.941176,0.184838,1249
2,0.25,0.097688,0.963235,0.177387,1341
1,0.20,0.095881,0.992647,0.174870,1408
0,0.15,0.093278,1.000000,0.170640,1458


In [59]:
fq_audit_path = OUTPUT_DIR / "prediabetes_final_quiz_feature_audit.csv"
fq_univariate_path = OUTPUT_DIR / "prediabetes_final_quiz_univariate_ranking.csv"
fq_perm_path = OUTPUT_DIR / "prediabetes_final_quiz_permutation_importance.csv"
fq_combined_path = OUTPUT_DIR / "prediabetes_final_quiz_candidates.csv"
fq_comparison_path = OUTPUT_DIR / "prediabetes_final_quiz_vs_other_runs_comparison.csv"
fq_threshold_path = OUTPUT_DIR / "prediabetes_final_quiz_threshold_scan.csv"

final_quiz_audit_df.to_csv(fq_audit_path, index=False)
fq_univariate_df.to_csv(fq_univariate_path, index=False)
perm_df_fq.to_csv(fq_perm_path, index=False)
combined_df_fq.to_csv(fq_combined_path, index=False)
comparison_test_fq.to_csv(fq_comparison_path, index=False)
threshold_df_fq.to_csv(fq_threshold_path, index=False)

print("Saved:")
print(fq_audit_path.resolve())
print(fq_univariate_path.resolve())
print(fq_perm_path.resolve())
print(fq_combined_path.resolve())
print(fq_comparison_path.resolve())
print(fq_threshold_path.resolve())

Saved:
C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_prediabetes\prediabetes_final_quiz_feature_audit.csv
C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_prediabetes\prediabetes_final_quiz_univariate_ranking.csv
C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_prediabetes\prediabetes_final_quiz_permutation_importance.csv
C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_prediabetes\prediabetes_final_quiz_candidates.csv
C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_prediabetes\prediabetes_final_quiz_vs_other_runs_comparison.csv
C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_prediabetes\prediabetes_final_quiz_threshold_scan.csv


# Age & Gender